# KUL-TN-20: Check SPF pointing 

### Setup notebook

In [ ]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# Configure figure in notebook6
%matplotlib notebook

### Imports

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PlatoSim libraries
import platosim.plot            as pt
import platosim.utilities       as ut
import platosim.referenceFrames as rf
from platosim.simulation import Simulation
from platosim.lightcurve import LightCurve
from platosim.matplotlibrc import setup_notebook
setup_notebook()

In [ ]:
# Load all data for a single star
sims = "/STER/platodata/PLATOSIM/simulations_PLATO-PL-KUL-TN-0020"

## Input star catalogue

In [ ]:
# Fetch the input catalogue
df0 = pd.read_feather(sims + '/inputfiles/starcat_P1_SPF_targets.ftr')
df0['id'] = np.arange(0,len(df0))
df0

In [ ]:
# Plot overall figure
fig, ax = pt.plotPlatoFOV('SPF', raStars=df0.ra, decStars=df0.dec, magStars=df0.mag, fs=20, system='icrs', showGroups=False)
ax.set_title('Input catalogue for P1 sample');

In [ ]:
fig, ax = pt.drawStarsInSkyAitoff(df0.ra, df0.dec, None, figsize=(9,7));

## Generate simulation table

In [ ]:
# Load overview table of simulations
df1 = pd.read_feather(f'{sims}/P1/table_simcat_P1.ftr')
df1.head()

In [ ]:
df_G1_Q23 = df1[(df1.G==1) & (df1.Q==23)]
df_G2_Q23 = df1[(df1.G==2) & (df1.Q==23)]
df_G3_Q23 = df1[(df1.G==3) & (df1.Q==23)]
df_G4_Q23 = df1[(df1.G==4) & (df1.Q==23)]

df_G1_Q24 = df1[(df1.G==1) & (df1.Q==24)]
df_G2_Q24 = df1[(df1.G==2) & (df1.Q==24)]
df_G3_Q24 = df1[(df1.G==3) & (df1.Q==24)]
df_G4_Q24 = df1[(df1.G==4) & (df1.Q==24)]

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10,10))

ax[1,0].plot(df_G1_Q23.xFP, df_G1_Q23.yFP, 'b.')
ax[1,1].plot(df_G2_Q23.xFP, df_G2_Q23.yFP, 'g.')
ax[0,1].plot(df_G3_Q23.xFP, df_G3_Q23.yFP, 'y.')
ax[0,0].plot(df_G4_Q23.xFP, df_G4_Q23.yFP, 'r.')

ax[1,0].set_title('Group 1')
ax[1,1].set_title('Group 2')
ax[0,1].set_title('Group 3')
ax[0,0].set_title('Group 4')

ax[0,0].set_xlim(-85, 85)
ax[1,0].set_xlim(-85, 85)
ax[1,1].set_xlim(-85, 85)
ax[0,1].set_xlim(-85, 85)

ax[0,0].set_ylim(-85, 85)
ax[1,0].set_ylim(-85, 85)
ax[1,1].set_ylim(-85, 85)
ax[0,1].set_ylim(-85, 85)

In [ ]:
plt.figure(figsize=(7,7))
x = 57.1
plt.plot(df_G1_Q23.xFP,     df_G1_Q23.yFP,     'b.', label='Group 1')
plt.plot(df_G2_Q23.xFP + x, df_G2_Q23.yFP,     'g.', label='Group 2')
plt.plot(df_G4_Q23.xFP,     df_G4_Q23.yFP + x, 'r.', label='Group 4')
plt.plot(df_G3_Q23.xFP + x, df_G3_Q23.yFP + x, 'y.', label='Group 3')
plt.title('P1 sample Q23')
plt.legend();

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10,10))

ax[1,0].plot(df_G1_Q24.xFP, df_G1_Q24.yFP, 'b.')
ax[1,1].plot(df_G2_Q24.xFP, df_G2_Q24.yFP, 'g.')
ax[0,1].plot(df_G3_Q24.xFP, df_G3_Q24.yFP, 'y.')
ax[0,0].plot(df_G4_Q24.xFP, df_G4_Q24.yFP, 'r.')

ax[1,0].set_title('Group 1')
ax[1,1].set_title('Group 2')
ax[0,1].set_title('Group 3')
ax[0,0].set_title('Group 4')

a = 90
ax[0,0].set_xlim(-a, a)
ax[1,0].set_xlim(-a, a)
ax[1,1].set_xlim(-a, a)
ax[0,1].set_xlim(-a, a)

ax[0,0].set_ylim(-a, a)
ax[1,0].set_ylim(-a, a)
ax[1,1].set_ylim(-a, a)
ax[0,1].set_ylim(-a, a)

In [ ]:
plt.figure(figsize=(7,7))
x = 57.1
plt.plot(df_G1_Q24.xFP,     df_G1_Q24.yFP,     'b.', label='Group 1')
plt.plot(df_G2_Q24.xFP + x, df_G2_Q24.yFP,     'g.', label='Group 2')
plt.plot(df_G4_Q24.xFP,     df_G4_Q24.yFP + x, 'r.', label='Group 4')
plt.plot(df_G3_Q24.xFP + x, df_G3_Q24.yFP + x, 'y.', label='Group 3')
plt.title('P1 sample Q24')
plt.legend();

In [ ]:
# Total number of (successful) simulations
N = len(df1)

# Convert to numpy
xFP = df1.xFP.to_numpy()
yFP = df1.yFP.to_numpy()

# Sky coordinates of pointing KUL20 [rad]
alpha, delta, kappa = ut.getPointingField('KUL20', unit='rad')

# Convert to arrays and change kappa as it depends on the quarter [rad]
alpha = np.ones(N) * alpha
delta = np.ones(N) * delta
kappa = np.fmod(df1.Q * np.pi/2 - np.pi/2, 2*np.pi) + kappa
kappa = kappa.to_numpy()

# Calculate camera azimuth and tilt
sim = Simulation('test')

# Calculate camera azimuth and tilt
azimuth = np.deg2rad([sim["CameraGroups/AzimuthAngle"][df1.G.iloc[i]-1] for i in range(N)])
tilt    = np.deg2rad([sim["CameraGroups/TiltAngle"][df1.G.iloc[i]-1]    for i in range(N)])

# Fetch focal plaen angle and focal length
angleFP     = np.zeros(N)
focalLength = np.array(sim["Camera/FocalLength/ConstantValue"] * 1000.)

# Re-calculate the sky positions as a sanity check
ra, dec = rf.focalPlaneToSkyCoordinates(xFP, yFP, alpha, delta, kappa, 
                                        tilt, azimuth, angleFP, focalLength)
# Convert to degrees
raDeg, decDeg = np.rad2deg(ra), np.rad2deg(dec)

In [ ]:
raDeg, decDeg

In [ ]:
# Plot overall figure
fig, ax = pt.plotPlatoFOV('SPF', raStars=raDeg, decStars=decDeg, magStars=None, 
                          fs=20, system='icrs', showGroups=False, fovSize=30)
ax.set_title('Ouput for P1 sample')
plt.show();
# Save figure
# fig.savefig('PlatoFOV.png', bbox_inches='tight', dpi=200)

In [ ]:
fig, ax = pt.drawStarsInSkyAitoff(raDeg, decDeg, None, figsize=(9,7))

In [ ]:
sim = Simulation('test')
alpha, delta, kappa = ut.getPointingField('KUL20')
df = df1.iloc[0]
fig = plt.figure(figsize=(12,10))
pt.drawStarInCCDfocalPlane(fig, sim, df.xCCD, df.yCCD, str(int(df.ccd)), int(df.group), alpha, delta, kappa)

## Check simulation statistics

In [ ]:
# Number of light curves per camera
ofile = 'starcat_P1_simulation.ftr'
lcs.statistic_stars_per_ncam(idir, ofile, numStar)

In [ ]:
df = pd.read_feather(ofile)

In [ ]:
# Number of light curves per group
df[['group', 'camera']].value_counts()

In [ ]:
# Number of light curves per group
df.value_counts()